In [10]:
import os
import glob
import mne
import numpy as np
from scipy.io import loadmat

dataset_dir = "/projects/EEG-foundation-model/RECH202/data/EEG"
eeg_dir = os.path.join(dataset_dir, "EEG")
logs_dir = os.path.join(dataset_dir, "logs")

eeg_files = sorted(glob.glob(os.path.join(eeg_dir, "*")))
log_files = sorted(glob.glob(os.path.join(logs_dir, "*")))
print(len(eeg_files), "EEG files,", len(log_files), "log files")

351 EEG files, 62 log files


In [11]:
def read_eeg(path):
    ext = os.path.splitext(path)[1].lower()
    readers = {
        ".vhdr": mne.io.read_raw_brainvision,
        ".edf": mne.io.read_raw_edf,
        ".bdf": mne.io.read_raw_bdf,
        ".fif": mne.io.read_raw_fif,
        ".set": mne.io.read_raw_eeglab,
        ".cnt": mne.io.read_raw_cnt,
        ".gdf": mne.io.read_raw_gdf,
    }
    return readers[ext](path, preload=True, verbose="ERROR")

In [12]:
header_files = [f for f in eeg_files if f.lower().endswith((".vhdr", ".edf", ".bdf", ".fif", ".set", ".cnt", ".gdf"))]
raw = read_eeg(header_files[0])
print(os.path.basename(header_files[0]))
print(raw.info)
print("sfreq:", raw.info["sfreq"], "| n_channels:", len(raw.ch_names), "| duration(s):", raw.n_times / raw.info["sfreq"])
print("channels:", raw.ch_names)
print("annotations:", set(raw.annotations.description))

001_DECOMEG_S1_10754_tapping.vhdr
<Info | 7 non-empty values
 bads: []
 ch_names: Fp1, Fp2, AF3, AF4, AF7, AF8, AFz, F1, F2, F3, F4, F5, F6, F7, ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 1000.0 Hz
 meas_date: 2022-11-22 12:22:18 UTC
 nchan: 64
 projs: []
 sfreq: 1000.0 Hz
>
sfreq: 1000.0 | n_channels: 64 | duration(s): 465.2
channels: ['Fp1', 'Fp2', 'AF3', 'AF4', 'AF7', 'AF8', 'AFz', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FT7', 'FT8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'T7', 'T8', 'Cz', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'TP7', 'TP8', 'CPz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'Pz', 'PO3', 'PO4', 'PO7', 'PO8', 'POz', 'O1', 'O2', 'Oz', 'M1', 'M2', 'RVEOG', 'LHEOG', 'RHEOG']
annotations: {np.str_('Stimulus/S213'), np.str_('Stimulus/S172'), np.str_('Stimulus/S124'), np.str_('Stimulus/S 31'), np.str_('Stimulus/S248'), np.str_('Stimulus/S113'), np.str_('Stimulus/S 13'), np.str_('Stimu

In [13]:
log = loadmat(log_files[0], squeeze_me=True, struct_as_record=False)
print(os.path.basename(log_files[0]))
for k, v in log.items():
    if k.startswith("__"):
        continue
    info = getattr(v, "_fieldnames", None) or getattr(v, "shape", type(v).__name__)
    print(k, "->", info)

S10_session1_block1_list1.mat
Pin1 -> int
a -> (37,)
angle -> float
angles -> float
ans -> float
b -> (35,)
baseRect -> (4,)
black -> (3,)
block -> int
blocklength -> int
blocknum -> int
c -> float
color -> (3,)
degPerFrame -> int
devnb -> (0,)
dim -> int
dir -> str
echo -> (0,)
endtraining1 -> int
endtraining2 -> int
errp -> (66,)
errperc -> float
eventkey -> ['Type', 'Time', 'Pressed', 'Keycode', 'CookedKey', 'ButtonStates', 'Motion', 'X', 'Y', 'NormX', 'NormY', 'Valuators']
fbblock -> int
fbtext -> str
fbtrial -> int
file -> int
filename -> str
flushwords -> int
gray -> float
ifi -> float
ii -> int
j -> int
jit -> float
k -> int
key -> (73,)
keyTrig -> (0,)
kp -> (37,)
line -> int
linestr -> str
list -> int
makebreak -> int
mask -> int
n -> int
noresp -> int
nremaining -> int
nrep -> int
ntrials -> int
ntrials1 -> int
order -> (64,)
order1 -> (136,)
origdir -> str
outtrial -> int
p -> int
password -> int
posX -> int
posXs -> int
posY -> int
posYs -> int
pr_trials -> ['order', 'seque

In [14]:
import re

raw = read_eeg(header_files[0])
events, event_id = mne.events_from_annotations(raw, verbose="ERROR")
code_to_ascii = {c: int(re.search(r"(\d+)", name).group(1)) for name, c in event_id.items() if re.search(r"S\s*\d+", name)}
keep = {c: a for c, a in code_to_ascii.items() if 32 <= a <= 126}
epochs = mne.Epochs(raw, events, event_id={f"key_{a}": c for c, a in keep.items()},
                    tmin=-0.2, tmax=0.5, baseline=None, preload=True, verbose="ERROR")
x = epochs.get_data()
y = np.array([keep[c] for c in epochs.events[:, 2]])
chars = np.array([chr(a) for a in y])
print("x:", x.shape, "| y:", y.shape)
print("labels:", sorted(set(chars)))

x: (225, 64, 701) | y: (225,)
labels: [np.str_(' '), np.str_('!'), np.str_('"'), np.str_('#'), np.str_('$'), np.str_('%'), np.str_('&'), np.str_("'"), np.str_('('), np.str_(')'), np.str_('*'), np.str_('+'), np.str_(','), np.str_('-'), np.str_('.'), np.str_('/'), np.str_('0'), np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'), np.str_('9'), np.str_(':'), np.str_(';'), np.str_('<'), np.str_('='), np.str_('>'), np.str_('?'), np.str_('@'), np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('J'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('O'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('X'), np.str_('Y'), np.str_('Z'), np.str_('['), np.str_('\\'), np.str_(']'), np.str_('^'), np.str_('_'), np.str_('`'), np.str_('a'), np.str_('b'), np.str_('c'), np.str_(

In [15]:
def load_xy(path):
    raw = read_eeg(path)
    events, event_id = mne.events_from_annotations(raw, verbose="ERROR")
    code_to_ascii = {c: int(re.search(r"(\d+)", name).group(1)) for name, c in event_id.items() if re.search(r"S\s*\d+", name)}
    keep = {c: a for c, a in code_to_ascii.items() if 32 <= a <= 126}
    if not keep:
        return None, None
    epochs = mne.Epochs(raw, events, event_id={f"key_{a}": c for c, a in keep.items()},
                        tmin=-0.2, tmax=0.5, baseline=None, preload=True, verbose="ERROR")
    return epochs.get_data(), np.array([keep[c] for c in epochs.events[:, 2]])

X, Y = [], []
for path in header_files[:5]:
    try:
        xi, yi = load_xy(path)
        if xi is None:
            continue
        X.append(xi)
        Y.append(yi)
        print(os.path.basename(path), xi.shape)
    except Exception as e:
        print("skip", os.path.basename(path), e)

X = np.concatenate(X)
Y = np.concatenate(Y)
print("X:", X.shape, "| Y:", Y.shape, "| labels:", sorted(set(chr(a) for a in Y)))

001_DECOMEG_S1_10754_tapping.vhdr (225, 64, 701)
001_DECOMEG_S1_10754_task1.vhdr (2201, 64, 701)
001_DECOMEG_S1_10754_task1bis.vhdr (2050, 64, 701)
001_DECOMEG_S1_10754_task2.vhdr (2917, 64, 701)
001_DECOMEG_S1_9949_task2.vhdr (668, 64, 701)
X: (8061, 64, 701) | Y: (8061,) | labels: [' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~']
